# List of Lagrangians


## Setup


### Imports


In [98]:
import re
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    ComplexScalarKineticTerm,
    CovD,
    DiracKineticTerm,
    Field,
    FieldStrength,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeKineticTerm,
    GaugeRepresentation,
    GhostLagrangian,
    Model,
    PartialD,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I, compact_sum_notation, compact_vertex_sum_form

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def expr_text(expr):
    text = expr.to_canonical_string() if hasattr(expr, "to_canonical_string") else str(expr)
    return text.replace("spenso::", "")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def signature_rows(signatures):
    return tuple(
        {
            "fields": ", ".join(signature.names),
            "arity": signature.arity,
            "terms": signature.term_count,
            "sectors": ", ".join(signature.sectors),
        }
        for signature in signatures
    )


def report_row(report):
    return {
        "matched_signatures": report.matched_signatures,
        "matched_terms": report.matched_terms,
        "total_signatures": report.total_signatures,
        "total_terms": report.total_terms,
        "signatures": signature_rows(report.signatures),
    }


def validation_rows(report):
    return {
        "ok": report.ok,
        "issues": tuple(
            {
                "code": issue.code,
                "severity": issue.severity,
                "message": issue.message,
            }
            for issue in report.issues
        ),
    }


def show(model, *, compact_form=None, sum_notation=None):
    print("==========")
    source_terms = model.source_lagrangian_terms()
    if source_terms:
        print("Lagrangian = ", clean(source_terms[0]))
    rules = model.lagrangian().feynman_rules(include_delta=False)
    print("Feynman Rule = ", clean(next(iter(rules.values()))))
    if compact_form is not None:
        print("Compact Form = ", clean(compact_form))
    if sum_notation is not None:
        print("Sum Notation = ", clean(sum_notation))
    print()


### Symbols


In [99]:
mu, nu, rho = S("mu"), S("nu"), S("rho")
eQED, gS, g2, xiQCD = S("eQED"), S("gS"), S("g2"), S("xiQCD")
qPhi, qPsi, qQ = S("qPhi"), S("qPsi"), S("qQ")
lam, y, g4, gGamma = S("lam"), S("y"), S("g4"), S("gGamma")
lam4, g, lamC, gD2, gijk = S("lam4"), S("g"), S("lamC"), S("gD2"), S("gijk")
i, j, k = S("i"), S("j"), S("k")


### Fields


In [100]:
Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = Field(
    "ghW",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
)
H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
)

# Compact scalar-only species for the short model-layer examples below.
PhiR = Field("PhiR", spin=0, self_conjugate=True, symbol=S("phi0"))
ChiR = Field("ChiR", spin=0, self_conjugate=True, symbol=S("chi0"))
PhiC = Field("PhiC", spin=0, self_conjugate=False, symbol=S("phiC0"), conjugate_symbol=S("phiCdag0"))
Phi_i = Field("phi_i", spin=0, self_conjugate=True, symbol=S("phi_i0"))
Phi_j = Field("phi_j", spin=0, self_conjugate=True, symbol=S("phi_j0"))
Phi_k = Field("phi_k", spin=0, self_conjugate=True, symbol=S("phi_k0"))
Phi4 = Field("Phi4", spin=0, self_conjugate=True, symbol=S("phi40"))

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, H, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers={'Q': qPhi}),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers={}),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers={'Q': qPsi}),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=xi, conjugate_symbol=xibar, mass=None, quantum_numbers={}),
 Field(name='q', spin=Fraction(1, 2), self_conjug

### Gauge Representations


In [101]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10b06c720>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10b06c880>, name='doublet', slot=None, slot_policy='unique'))

### Gauge Groups


In [102]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon.symbol,
    charge="Q",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon.symbol,
    ghost_field=GhostG.symbol,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W.symbol,
    ghost_field=GhostW.symbol,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=A0, ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=G0, ghost_field=ghG0, structure_constant=<function structure_constant at 0x10b06c7d0>, representations=(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10b06c720>, name='fund', slot=None, slot_policy='unique'),), charge=None),
 GaugeGroup(name='SU2L', abelian=False, coupling=g2, gauge_boson=W0, ghost_field=ghW0, structure_constant=<function weak_structure_constant at 0x10b06c930>, representations=(GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10b06c880>, name='

## Scalar Model-Layer Examples


In [103]:
phi4 = Model(fields=(PhiR,), lagrangian_decl=lam4 * PhiR * PhiR * PhiR * PhiR)
show(phi4)

phi2chi2 = Model(fields=(PhiR, ChiR), lagrangian_decl=g * PhiR * PhiR * ChiR * ChiR)
show(phi2chi2)

complex_bilinear = Model(fields=(PhiC,), lagrangian_decl=lamC * PhiC.bar * PhiC)
show(complex_bilinear)

multi_species = Model(
    fields=(Phi_i, Phi_j, Phi_k),
    lagrangian_decl=gijk(i, j, k) * Phi_i * Phi_i * Phi_j * Phi_j * Phi_k * Phi_k,
)
show(multi_species)


Lagrangian =  lam4 * PhiR * PhiR * PhiR * PhiR
Feynman Rule =  24𝑖*lam4

Lagrangian =  g * PhiR * PhiR * ChiR * ChiR
Feynman Rule =  4𝑖*g

Lagrangian =  lamC * PhiC.bar * PhiC
Feynman Rule =  1𝑖*lamC

Lagrangian =  gijk(i,j,k) * phi_i * phi_i * phi_j * phi_j * phi_k * phi_k
Feynman Rule =  8𝑖*gijk(i,j,k)



In [115]:

derivative_phi4 = Model(
    fields=(PhiR,),
    lagrangian_decl=gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR,
)
show(derivative_phi4)

Lagrangian =  gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR
Feynman Rule =  -4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)



## Local Models


In [116]:
local_yukawa_model = Model(fields=(Psi, Phi), lagrangian_decl=y * Psi.bar * Psi * Phi)
local_four_fermion_model = Model(fields=(Psi, Xi), lagrangian_decl=g4 * Psi.bar * Psi * Xi.bar * Xi)
local_quartic_model = Model(fields=(Phi, Chi), lagrangian_decl=lam * Phi.bar * Phi * Chi.bar * Chi)
gamma_model = Model(
    fields=(Psi,),
    lagrangian_decl=(
        gGamma * Psi.bar * Gamma(mu) * Gamma(nu) * Psi
        + gGamma * Psi.bar * Gamma(nu) * Gamma(mu) * Psi
    ),
)
